In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from umap import UMAP
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
EMBEDDINGS_DIR = PROJECT_ROOT / "data" / "glomeruli" / "embeddings"

# Input modificabili
DATASET_PATHS = {
    "masked": EMBEDDINGS_DIR / "nasnet_masked_embeddings.npy",
    "unmasked": EMBEDDINGS_DIR / "nasnet_unmasked_embeddings.npy",
}
PCA_VARIANCES = 0.95
RANDOM_STATE = 42

In [ ]:
def fit_pca(x, n_components = 0.95):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(x)
    pca = PCA(n_components=n_components)
    return pca.fit_transform(scaled)

def fit_umap(
        x,
        n_components,
        n_neighbors = 15,
        random_state = RANDOM_STATE,
        min_dist = 0.1,
):
    umap = UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        random_state=random_state,
        metric="euclidean",
    )

    return umap.fit_transform(x)

def plot_pca_umap_square(
    masked_pca,
    unmasked_pca,
    masked_umap,
    unmasked_umap,
    labels_masked=None,
    labels_unmasked=None,
    s=5,
    alpha=0.7,
    cmap="viridis"
):
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))

    plots = [
        (masked_pca,   labels_masked,   "Masked PCA",   axes[0, 0]),
        (unmasked_pca, labels_unmasked, "Unmasked PCA", axes[0, 1]),
        (masked_umap,  labels_masked,   "Masked UMAP",  axes[1, 0]),
        (unmasked_umap,labels_unmasked, "Unmasked UMAP",axes[1, 1]),
    ]

    for emb, labels, title, ax in plots:
        if labels is not None:
            sc = ax.scatter(
                emb[:, 0],
                emb[:, 1],
                c=labels,
                s=s,
                alpha=alpha,
                cmap=cmap,
                linewidths=0
            )
        else:
            ax.scatter(
                emb[:, 0],
                emb[:, 1],
                s=s,
                alpha=alpha,
                linewidths=0
            )

        ax.set_title(title)
        ax.set_xlabel("Component 1")
        ax.set_ylabel("Component 2")
        ax.set_aspect("equal", adjustable="datalim")

    plt.tight_layout()
    plt.show()

In [ ]:
masked = np.load(DATASET_PATHS["masked"])
unmasked = np.load(DATASET_PATHS["unmasked"])

masked_pca = fit_pca(masked)
unmasked_pca = fit_pca(unmasked)

masked_umap = fit_umap(masked, 2)
unmasked_umap = fit_umap(unmasked, 2)

plot_pca_umap_square(
    masked_pca,
    unmasked_pca,
    masked_umap,
    unmasked_umap
)


In [ ]:
import plotly.graph_objects as go


def plot_umap_3d(emb_umap, title):
    """
    Visualizza un embedding UMAP 3D in modo dinamico/interattivo.

    emb_umap deve avere shape:
    (numero_immagini, 3)
    """

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=emb_umap[:, 0],
                y=emb_umap[:, 1],
                z=emb_umap[:, 2],
                mode="markers",
                marker=dict(
                    size=4,
                    opacity=0.75
                )
            )
        ]
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="UMAP 1",
            yaxis_title="UMAP 2",
            zaxis_title="UMAP 3"
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.show()

In [ ]:
plot_umap_3d(unmasked_umap_3, "Unmasked glomeruli")dinov2_large_patchmasked_umap_3 = fit_umap(masked, 3)
unmasked_umap_3 = fit_umap(unmasked, 3)

plot_umap_3d(masked_umap_3, "Masked glomeruli")


In [ ]:
plot_umap_3d(unmasked_umap_3, "Unmasked glomeruli")